# FAME-Energy — Temporal Replication v3.2
## Robustness to operating-capacity margins

Esta versão é uma extensão da v3.1 e responde a uma questão metodológica específica:

> Os resultados de transferibilidade temporal do FAME permanecem quando alteramos a folga de capacidade do problema decisório?

Na v3.1, a escala colocava aproximadamente a carga média do Development no mesmo nível da capacidade térmica disponível. Isso pode tornar o sistema artificialmente apertado e aumentar o valor de representações conservadoras.

A v3.2 mantém **inalterados**:

- as nove replicações temporais;
- as três fontes preditivas: ENTSO-E, SARIMAX e LSTM;
- a seleção dos modelos preditivos no Development;
- a transformação FAME;
- o problema binário de seleção de capacidade;
- o protocolo Calibration → freeze → Test.

A única dimensão adicional é a **margem média de capacidade**:

\[
M\in\{1.00,1.10,1.20,1.30,1.40\}.
\]

Para cada replicação \(k\), a escala é definida por

\[
s_{k,M}
=
\frac{K}
{M\,\overline D_{\mathrm{Dev},k}},
\]

onde \(K\) é a capacidade térmica total. Portanto,

\[
\frac{K}
{\overline D_{\mathrm{Dev},k}^{scaled}}
=
M.
\]

Assim, \(M=1.20\) significa que a capacidade térmica total equivale a aproximadamente 120% da carga média escalada do Development.

---

## Duas análises de robustez

### A. DOC recalibrado dentro de cada margem

Para cada \(M\),

\[
\widehat\theta_{m,k,M}
=
\arg\min_\theta
\overline L_{\mathrm{Cal},m,k,M}(\theta).
\]

Isso responde:

> O mecanismo FAME continua transferindo quando o regime operacional muda?

### B. Stress test com \(\theta\) congelado na margem de referência

Primeiro calibramos na referência \(M=1.00\):

\[
\widehat\theta^{ref}_{m,k}.
\]

Depois aplicamos **o mesmo \(\widehat\theta^{ref}_{m,k}\)** aos cenários \(M>1\).

Isso responde uma pergunta diferente e mais forte:

> A representação aprendida continua útil quando muda o ambiente decisório, sem recalibração?

As duas análises são reportadas separadamente.

## 1. Dependências e instalação automática

In [ ]:
import importlib.util, subprocess, sys

REQUIRED = {
    "numpy":"numpy",
    "pandas":"pandas",
    "matplotlib":"matplotlib",
    "sklearn":"scikit-learn",
    "statsmodels":"statsmodels",
    "pulp":"pulp",
    "tensorflow":"tensorflow",
}

installed_now=[]

for module,package in REQUIRED.items():
    if importlib.util.find_spec(module) is None:
        print("Instalando",package)
        subprocess.check_call([sys.executable,"-m","pip","install",package])
        installed_now.append(package)

if installed_now:
    print("Instalados:",installed_now)
    print("Se necessário, reinicie o kernel e execute novamente desde o início.")
else:
    print("Dependências: OK")

In [ ]:
from pathlib import Path
import json, hashlib, random, warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.metrics import mean_absolute_error, mean_squared_error
from sklearn.preprocessing import StandardScaler
from statsmodels.tsa.statespace.sarimax import SARIMAX

import pulp
import tensorflow as tf
from tensorflow.keras.layers import Input, LSTM, Dense, Concatenate
from tensorflow.keras.models import Model
from tensorflow.keras.callbacks import EarlyStopping
from tensorflow.keras.optimizers import Adam

warnings.filterwarnings("ignore")

SEED=123
random.seed(SEED)
np.random.seed(SEED)
tf.random.set_seed(SEED)

ROOT=Path.cwd()
OUT=ROOT/"data"/"fame_energy_temporal_replication_v32_robustness"
OUT.mkdir(parents=True,exist_ok=True)

CBC_EXE=Path(
    r"C:\Users\tiago\Dropbox\Artigos\FAME\FAME_IJDSA"
    r"\Cbc-releases.2.10.13-windows-2025-msvs-v17-Release-x64"
    r"\bin\cbc.exe"
)

VOLL=10_000.0
EMERGENCY_COST_MULTIPLIER=10.0

THETA_GRID=np.array(
    [-0.10,-0.075,-0.05,-0.025,0.0,0.025,0.05,0.075,0.10],
    dtype=float
)

CAPACITY_MARGINS=np.array([1.00,1.10,1.20,1.30,1.40],dtype=float)

print("Output:",OUT)
print("Margins:",CAPACITY_MARGINS.tolist())

## 2. Série canônica TransnetBW/ENTSO-E 2015–2025

In [ ]:
candidates=[
    ROOT/"data"/"prepared"/"fame_energy_v2"/"transnetbw_actual_forecast_hourly_utc_2015_2025.csv",
    ROOT/"data"/"prepared"/"transnetbw"/"transnetbw_actual_forecast_hourly_utc_2015_2025.csv",
]

canonical=next((p for p in candidates if p.exists()),None)

if canonical is None:
    found=list(ROOT.rglob("transnetbw_actual_forecast_hourly_utc_2015_2025.csv"))
    canonical=found[0] if found else None

if canonical is None:
    raise FileNotFoundError(
        "Arquivo canônico TransnetBW 2015–2025 não encontrado."
    )

z=pd.read_csv(canonical)
z["timestamp_utc"]=pd.to_datetime(z["timestamp_utc"],utc=True)

required={"timestamp_utc","actual_load_mw","dayahead_forecast_mw"}
missing=required.difference(z.columns)
if missing:
    raise RuntimeError(f"Colunas ausentes: {sorted(missing)}")

z["date"]=z["timestamp_utc"].dt.floor("D")

daily=(
    z.groupby("date",as_index=False)
     .agg(
         actual_peak=("actual_load_mw","max"),
         entsoe_forecast_peak=("dayahead_forecast_mw","max"),
         n_hours=("timestamp_utc","size"),
     )
     .sort_values("date")
     .reset_index(drop=True)
)

daily=daily[daily["n_hours"]>=23].copy()

daily["year"]=daily["date"].dt.year
daily["dow"]=daily["date"].dt.dayofweek
daily["dow_sin"]=np.sin(2*np.pi*daily["dow"]/7)
daily["dow_cos"]=np.cos(2*np.pi*daily["dow"]/7)
daily["doy"]=daily["date"].dt.dayofyear
daily["doy_sin"]=np.sin(2*np.pi*daily["doy"]/366)
daily["doy_cos"]=np.cos(2*np.pi*daily["doy"]/366)

print("Arquivo:",canonical)
print("Anos:",sorted(daily.year.unique()))
print("Dias:",len(daily))

## 3. Nove replicações temporais

In [ ]:
replications=[]

for test_year in range(2017,2026):
    cal_year=test_year-1
    dev_years=list(range(2015,cal_year))

    replications.append({
        "replication_id":f"E{test_year}",
        "development_start":min(dev_years),
        "development_end":max(dev_years),
        "calibration_year":cal_year,
        "test_year":test_year,
    })

replications=pd.DataFrame(replications)

needed=set(range(2015,2026))
available=set(daily.year.unique())
if not needed.issubset(available):
    raise RuntimeError(
        f"Anos ausentes: {sorted(needed.difference(available))}"
    )

display(replications)

## 4. Funções preditivas

In [ ]:
EXOG_COLS=["dow_sin","dow_cos","doy_sin","doy_cos"]
CAL_COLS=EXOG_COLS

def regression_metrics(y_true,y_pred):
    y_true=np.asarray(y_true,dtype=float)
    y_pred=np.asarray(y_pred,dtype=float)
    return {
        "rmse":float(np.sqrt(mean_squared_error(y_true,y_pred))),
        "mae":float(mean_absolute_error(y_true,y_pred)),
        "mape":float(np.mean(
            np.abs(y_true-y_pred)/np.clip(np.abs(y_true),1e-9,None)
        )),
        "bias":float(np.mean(y_pred-y_true)),
    }

SARIMAX_CANDIDATES=[
    {"order":(1,0,0),"seasonal_order":(0,0,0,7)},
    {"order":(1,0,1),"seasonal_order":(0,0,0,7)},
    {"order":(2,0,0),"seasonal_order":(0,0,0,7)},
    {"order":(1,0,0),"seasonal_order":(1,0,0,7)},
    {"order":(1,0,1),"seasonal_order":(1,0,0,7)},
]

def select_sarimax(dev):
    dev=dev.sort_values("date").reset_index(drop=True)
    split=max(60,int(len(dev)*0.80))
    split=min(split,len(dev)-30)

    tr=dev.iloc[:split]
    va=dev.iloc[split:]
    rows=[]

    for cfg in SARIMAX_CANDIDATES:
        try:
            fit=SARIMAX(
                tr["actual_peak"].values,
                exog=tr[EXOG_COLS].values,
                order=cfg["order"],
                seasonal_order=cfg["seasonal_order"],
                trend="c",
                enforce_stationarity=False,
                enforce_invertibility=False
            ).fit(disp=False,maxiter=150)

            pred=fit.get_forecast(
                steps=len(va),
                exog=va[EXOG_COLS].values
            ).predicted_mean

            met=regression_metrics(va["actual_peak"],pred)
            rows.append({**cfg,**met,"aic":float(fit.aic)})

        except Exception:
            rows.append({
                **cfg,"rmse":np.inf,"mae":np.inf,"mape":np.inf,
                "bias":np.nan,"aic":np.inf
            })

    tab=pd.DataFrame(rows)
    ok=tab[np.isfinite(tab.rmse)]
    if ok.empty:
        raise RuntimeError("Nenhum SARIMAX convergiu.")

    best=ok.sort_values(["rmse","mae","aic"]).iloc[0]
    return tuple(best["order"]),tuple(best["seasonal_order"]),tab

def sarimax_sequential_forecast(dev,future,order,seasonal_order):
    fit=SARIMAX(
        dev["actual_peak"].values,
        exog=dev[EXOG_COLS].values,
        order=order,
        seasonal_order=seasonal_order,
        trend="c",
        enforce_stationarity=False,
        enforce_invertibility=False
    ).fit(disp=False,maxiter=200)

    preds=[]

    for row in future.sort_values("date").itertuples(index=False):
        ex=np.array([[getattr(row,c) for c in EXOG_COLS]],dtype=float)
        pred=float(fit.get_forecast(steps=1,exog=ex).predicted_mean[0])
        preds.append(pred)

        fit=fit.append(
            endog=[float(row.actual_peak)],
            exog=ex,
            refit=False
        )

    return np.asarray(preds)

In [ ]:
LSTM_CANDIDATES=[
    {"lookback":7,"units":16,"lr":1e-3},
    {"lookback":14,"units":16,"lr":1e-3},
    {"lookback":14,"units":32,"lr":1e-3},
    {"lookback":21,"units":32,"lr":5e-4},
]

def make_sequences(df,lookback):
    df=df.sort_values("date").reset_index(drop=True)
    vals=df["actual_peak"].values.astype(float)

    Xs,Xc,y=[],[],[]
    for i in range(lookback,len(df)):
        Xs.append(vals[i-lookback:i].reshape(-1,1))
        Xc.append(df.loc[i,CAL_COLS].values.astype(float))
        y.append(vals[i])

    return np.asarray(Xs),np.asarray(Xc),np.asarray(y)

def build_lstm(lookback,units,lr):
    seq_in=Input(shape=(lookback,1))
    cal_in=Input(shape=(len(CAL_COLS),))
    x=LSTM(units)(seq_in)
    x=Concatenate()([x,cal_in])
    x=Dense(16,activation="relu")(x)
    out=Dense(1)(x)

    model=Model([seq_in,cal_in],out)
    model.compile(optimizer=Adam(learning_rate=lr),loss="mse")
    return model

def select_lstm(dev):
    rows=[]

    for cfg in LSTM_CANDIDATES:
        Xs,Xc,y=make_sequences(dev,int(cfg["lookback"]))

        split=max(30,int(len(y)*0.80))
        split=min(split,len(y)-20)
        if split<=0 or len(y)-split<10:
            continue

        Xs_tr,Xs_va=Xs[:split],Xs[split:]
        Xc_tr,Xc_va=Xc[:split],Xc[split:]
        y_tr,y_va=y[:split],y[split:]

        scaler=StandardScaler().fit(y_tr.reshape(-1,1))
        mu,sd=scaler.mean_[0],scaler.scale_[0]

        Xs_tr_sc=(Xs_tr-mu)/sd
        Xs_va_sc=(Xs_va-mu)/sd
        y_tr_sc=scaler.transform(y_tr.reshape(-1,1)).ravel()
        y_va_sc=scaler.transform(y_va.reshape(-1,1)).ravel()

        tf.keras.backend.clear_session()
        tf.random.set_seed(SEED)

        model=build_lstm(
            int(cfg["lookback"]),int(cfg["units"]),float(cfg["lr"])
        )

        es=EarlyStopping(
            monitor="val_loss",patience=15,restore_best_weights=True
        )

        hist=model.fit(
            [Xs_tr_sc,Xc_tr],y_tr_sc,
            validation_data=([Xs_va_sc,Xc_va],y_va_sc),
            epochs=200,batch_size=32,verbose=0,callbacks=[es]
        )

        pred_sc=model.predict([Xs_va_sc,Xc_va],verbose=0).ravel()
        pred=scaler.inverse_transform(pred_sc.reshape(-1,1)).ravel()

        rows.append({
            **cfg,
            **regression_metrics(y_va,pred),
            "epochs":len(hist.history["loss"])
        })

    tab=pd.DataFrame(rows)
    if tab.empty:
        raise RuntimeError("Nenhuma LSTM avaliada.")

    best=tab.sort_values(["rmse","mae"]).iloc[0]

    return {
        "lookback":int(best["lookback"]),
        "units":int(best["units"]),
        "lr":float(best["lr"])
    },tab

def fit_lstm_final(dev,cfg):
    Xs,Xc,y=make_sequences(dev,cfg["lookback"])

    scaler=StandardScaler().fit(y.reshape(-1,1))
    mu,sd=scaler.mean_[0],scaler.scale_[0]

    tf.keras.backend.clear_session()
    tf.random.set_seed(SEED)

    model=build_lstm(cfg["lookback"],cfg["units"],cfg["lr"])

    es=EarlyStopping(
        monitor="loss",patience=20,restore_best_weights=True
    )

    model.fit(
        [(Xs-mu)/sd,Xc],
        scaler.transform(y.reshape(-1,1)).ravel(),
        epochs=250,batch_size=32,verbose=0,callbacks=[es]
    )

    return model,scaler

def lstm_sequential_forecast(model,scaler,history,future,lookback):
    observed=history.sort_values("date")["actual_peak"].astype(float).tolist()
    mu,sd=scaler.mean_[0],scaler.scale_[0]
    preds=[]

    for row in future.sort_values("date").itertuples(index=False):
        seq=np.asarray(observed[-lookback:],dtype=float).reshape(1,lookback,1)
        cal=np.array([[getattr(row,c) for c in CAL_COLS]],dtype=float)

        pred_sc=model.predict([(seq-mu)/sd,cal],verbose=0).ravel()[0]
        pred=float(
            scaler.inverse_transform(np.array([[pred_sc]]))[0,0]
        )
        preds.append(pred)

        observed.append(float(row.actual_peak))

    return np.asarray(preds)

## 5. Frota RTS-GMLC e problema decisório

In [ ]:
gen_candidates=list((ROOT/"external").rglob("SourceData/gen.csv"))
if not gen_candidates:
    gen_candidates=list(ROOT.rglob("SourceData/gen.csv"))
if not gen_candidates:
    raise FileNotFoundError("SourceData/gen.csv não encontrado.")

GEN_FILE=sorted(gen_candidates,key=lambda p:len(str(p)))[0]
gen=pd.read_csv(GEN_FILE)

thermal_mask=~gen["Fuel"].astype(str).isin(
    ["Wind","Solar","Storage","Hydro","CSP"]
)

generators=gen.loc[thermal_mask].copy()
generators["PMax MW"]=pd.to_numeric(generators["PMax MW"],errors="coerce")
generators=generators[
    generators["PMax MW"].notna()&(generators["PMax MW"]>0)
].reset_index(drop=True)

def representative_cost(row):
    pmax=float(row["PMax MW"])
    fuel=pd.to_numeric(pd.Series([row.get("Fuel Price $/MMBTU",np.nan)]),errors="coerce").iloc[0]
    op=pd.to_numeric(pd.Series([row.get("Output_pct_3",np.nan)]),errors="coerce").iloc[0]
    hr=pd.to_numeric(pd.Series([row.get("HR_incr_3",np.nan)]),errors="coerce").iloc[0]

    fuel=1.0 if pd.isna(fuel) or fuel<=0 else float(fuel)
    op=1.0 if pd.isna(op) or op<=0 else float(op)
    hr=10_000.0 if pd.isna(hr) or hr<=0 else float(hr)

    return max(pmax*op*hr*fuel/1000.0,1e-6)

generators["capacity_mw"]=generators["PMax MW"].astype(float)
generators["decision_cost"]=generators.apply(representative_cost,axis=1)
generators["g"]=np.arange(len(generators),dtype=int)

generator_table=generators[
    ["g","Fuel","capacity_mw","decision_cost"]
].copy()

generator_table["cost_per_mw"] = (
    generator_table["decision_cost"] /
    generator_table["capacity_mw"].clip(lower=1e-9)
)

TOTAL_CONVENTIONAL_CAPACITY=float(generator_table["capacity_mw"].sum())
EMERGENCY_COST_PER_MW=float(
    EMERGENCY_COST_MULTIPLIER*generator_table["cost_per_mw"].max()
)

print("Thermal capacity:",TOTAL_CONVENTIONAL_CAPACITY)
print("Emergency cost/MW:",EMERGENCY_COST_PER_MW)

In [ ]:
def solve_capacity_plan(required_capacity_mw):
    required_capacity_mw=float(required_capacity_mw)

    emergency_upper=max(
        0.0,
        required_capacity_mw-TOTAL_CONVENTIONAL_CAPACITY
    )

    problem=pulp.LpProblem("FAME_Energy_Capacity",pulp.LpMinimize)

    x={
        int(r.g):pulp.LpVariable(f"x_{int(r.g)}",0,1,cat=pulp.LpBinary)
        for r in generator_table.itertuples(index=False)
    }

    emergency=pulp.LpVariable(
        "emergency_mw",lowBound=0,upBound=emergency_upper,cat=pulp.LpContinuous
    )

    problem += (
        pulp.lpSum(
            float(r.decision_cost)*x[int(r.g)]
            for r in generator_table.itertuples(index=False)
        )
        + EMERGENCY_COST_PER_MW*emergency
    )

    problem += (
        pulp.lpSum(
            float(r.capacity_mw)*x[int(r.g)]
            for r in generator_table.itertuples(index=False)
        )
        + emergency
        >= required_capacity_mw
    )

    solver=(
        pulp.COIN_CMD(path=str(CBC_EXE),msg=False)
        if CBC_EXE.is_file()
        else pulp.PULP_CBC_CMD(msg=False)
    )

    status=problem.solve(solver)

    if pulp.LpStatus[status]!="Optimal":
        return None

    selected=[
        int(r.g)
        for r in generator_table.itertuples(index=False)
        if x[int(r.g)].value() is not None and x[int(r.g)].value()>0.5
    ]

    sel=generator_table[generator_table["g"].isin(selected)]
    emergency_mw=float(emergency.value() or 0.0)

    conventional_capacity=float(sel["capacity_mw"].sum())
    conventional_cost=float(sel["decision_cost"].sum())
    emergency_cost=EMERGENCY_COST_PER_MW*emergency_mw

    return {
        "available_capacity_mw":conventional_capacity+emergency_mw,
        "emergency_capacity_mw":emergency_mw,
        "decision_cost":conventional_cost+emergency_cost,
        "n_generators":len(selected),
    }

def evaluate_decision(date,model,forecast_peak,actual_peak,theta):
    required=float(forecast_peak)*np.exp(float(theta))
    plan=solve_capacity_plan(required)

    if plan is None:
        return {
            "date":date,"model":model,"theta":float(theta),
            "status":"solver_failure","realized_loss":np.inf
        }

    shortage=max(
        0.0,
        float(actual_peak)-plan["available_capacity_mw"]
    )

    return {
        "date":date,
        "model":model,
        "theta":float(theta),
        "forecast_peak":float(forecast_peak),
        "actual_peak":float(actual_peak),
        "required_capacity_mw":required,
        "available_capacity_mw":plan["available_capacity_mw"],
        "emergency_capacity_mw":plan["emergency_capacity_mw"],
        "decision_cost":plan["decision_cost"],
        "n_generators":plan["n_generators"],
        "shortage_mw":shortage,
        "realized_loss":plan["decision_cost"]+VOLL*shortage,
        "status":"optimal"
    }

## 6. Geração das previsões uma única vez por replicação

As previsões são geradas na escala original TransnetBW e reutilizadas em todas as margens.
Dessa forma, a análise de robustez altera somente o ambiente decisório, não a camada preditiva.

In [ ]:
prediction_cache={}
predictive_rows=[]
model_selection_rows=[]

for rep in replications.itertuples(index=False):
    rid=rep.replication_id
    print("\n",rid)

    dev=daily[daily.year.between(rep.development_start,rep.development_end)].copy()
    cal=daily[daily.year.eq(rep.calibration_year)].copy()
    tst=daily[daily.year.eq(rep.test_year)].copy()

    sar_order,sar_seasonal,sar_tab=select_sarimax(dev)
    lstm_cfg,lstm_tab=select_lstm(dev)
    lstm_model,lstm_scaler=fit_lstm_final(dev,lstm_cfg)

    sar_cal=sarimax_sequential_forecast(dev,cal,sar_order,sar_seasonal)
    lstm_cal=lstm_sequential_forecast(
        lstm_model,lstm_scaler,dev,cal,lstm_cfg["lookback"]
    )

    dev_cal=pd.concat([dev,cal],ignore_index=True).sort_values("date")

    sar_test=sarimax_sequential_forecast(
        dev_cal,tst,sar_order,sar_seasonal
    )
    lstm_test=lstm_sequential_forecast(
        lstm_model,lstm_scaler,dev_cal,tst,lstm_cfg["lookback"]
    )

    cal_pred=cal[["date","actual_peak","entsoe_forecast_peak"]].copy()
    cal_pred=cal_pred.rename(columns={"entsoe_forecast_peak":"ENTSOE"})
    cal_pred["SARIMAX"]=sar_cal
    cal_pred["LSTM"]=lstm_cal

    tst_pred=tst[["date","actual_peak","entsoe_forecast_peak"]].copy()
    tst_pred=tst_pred.rename(columns={"entsoe_forecast_peak":"ENTSOE"})
    tst_pred["SARIMAX"]=sar_test
    tst_pred["LSTM"]=lstm_test

    prediction_cache[rid]={
        "dev_mean_actual":float(dev["actual_peak"].mean()),
        "cal":cal_pred,
        "test":tst_pred,
        "sarimax_order":sar_order,
        "sarimax_seasonal":sar_seasonal,
        "lstm_cfg":lstm_cfg,
    }

    for stage,df in [("calibration",cal_pred),("test",tst_pred)]:
        for model in ["ENTSOE","SARIMAX","LSTM"]:
            predictive_rows.append({
                "replication_id":rid,
                "stage":stage,
                "model":model,
                **regression_metrics(df["actual_peak"],df[model])
            })

    model_selection_rows.append({
        "replication_id":rid,
        "sarimax_order":str(sar_order),
        "sarimax_seasonal_order":str(sar_seasonal),
        "lstm_lookback":lstm_cfg["lookback"],
        "lstm_units":lstm_cfg["units"],
        "lstm_lr":lstm_cfg["lr"],
    })

predictive=pd.DataFrame(predictive_rows)
model_selection=pd.DataFrame(model_selection_rows)

predictive.to_csv(OUT/"predictive_metrics_native_scale.csv",index=False)
model_selection.to_csv(OUT/"development_model_selection.csv",index=False)

print("Prediction cache complete.")

# Parte A — DOC recalibrado por margem

## 7. Calibration e Test para todos os cenários de margem

In [ ]:
cal_surface_rows=[]
test_rows=[]
freeze_rows=[]

for rep in replications.itertuples(index=False):
    rid=rep.replication_id
    cache=prediction_cache[rid]

    for margin in CAPACITY_MARGINS:
        scale=(
            TOTAL_CONVENTIONAL_CAPACITY /
            (float(margin)*cache["dev_mean_actual"])
        )

        cal=cache["cal"].copy()
        tst=cache["test"].copy()

        for c in ["actual_peak","ENTSOE","SARIMAX","LSTM"]:
            cal[c]=cal[c]*scale
            tst[c]=tst[c]*scale

        theta_map={}

        for model in ["ENTSOE","SARIMAX","LSTM"]:
            model_cal=[]

            for theta in THETA_GRID:
                rows=[
                    evaluate_decision(
                        row.date,model,getattr(row,model),row.actual_peak,float(theta)
                    )
                    for row in cal.itertuples(index=False)
                ]

                df=pd.DataFrame(rows)
                if not np.isfinite(df["realized_loss"]).all():
                    raise RuntimeError(
                        f"{rid}, M={margin}, {model}: perda não finita."
                    )

                cal_surface_rows.append({
                    "replication_id":rid,
                    "margin":float(margin),
                    "model":model,
                    "theta":float(theta),
                    "mean_loss":float(df["realized_loss"].mean()),
                    "total_shortage_mw":float(df["shortage_mw"].sum()),
                    "total_emergency_mw":float(df["emergency_capacity_mw"].sum()),
                    "mean_decision_cost":float(df["decision_cost"].mean()),
                })

                model_cal.append({
                    "theta":float(theta),
                    "mean_loss":float(df["realized_loss"].mean())
                })

            s=pd.DataFrame(model_cal)
            best_loss=s["mean_loss"].min()
            cand=s[np.isclose(s["mean_loss"],best_loss,rtol=1e-10,atol=1e-8)].copy()
            cand["abs_theta"]=cand["theta"].abs()
            theta_doc=float(cand.sort_values(["abs_theta","theta"]).iloc[0]["theta"])
            theta_map[model]=theta_doc

            for theta,strategy in [(0.0,"baseline"),(theta_doc,"FAME-DOC")]:
                for row in tst.itertuples(index=False):
                    rr=evaluate_decision(
                        row.date,model,getattr(row,model),row.actual_peak,float(theta)
                    )
                    rr["replication_id"]=rid
                    rr["margin"]=float(margin)
                    rr["strategy"]=strategy
                    test_rows.append(rr)

        freeze_rows.append({
            "replication_id":rid,
            "margin":float(margin),
            "scale_factor":float(scale),
            **{f"theta_{m}":v for m,v in theta_map.items()}
        })

    print(rid,"completed")

cal_surface=pd.DataFrame(cal_surface_rows)
test_decisions=pd.DataFrame(test_rows)
freezes=pd.DataFrame(freeze_rows)

cal_surface.to_csv(OUT/"calibration_surface_by_margin.csv",index=False)
test_decisions.to_csv(OUT/"test_decisions_by_margin.csv",index=False)
freezes.to_csv(OUT/"theta_freezes_by_margin.csv",index=False)

## 8. Transfer rate por modelo e margem — DOC recalibrado

In [ ]:
test_summary=(
    test_decisions.groupby(
        ["replication_id","margin","model","strategy"],as_index=False
    )
    .agg(
        mean_loss=("realized_loss","mean"),
        total_shortage_mw=("shortage_mw","sum"),
        total_emergency_mw=("emergency_capacity_mw","sum"),
    )
)

wide=test_summary.pivot(
    index=["replication_id","margin","model"],
    columns="strategy",
    values="mean_loss"
).reset_index()

wide["gain"]=wide["baseline"]-wide["FAME-DOC"]
wide["gain_pct"]=wide["gain"]/wide["baseline"]*100.0
wide["transferred"]=wide["gain"]>0

robustness_summary=(
    wide.groupby(["margin","model"],as_index=False)
    .agg(
        n_replications=("replication_id","count"),
        transfer_rate=("transferred","mean"),
        mean_gain_pct=("gain_pct","mean"),
        median_gain_pct=("gain_pct","median"),
        min_gain_pct=("gain_pct","min"),
        max_gain_pct=("gain_pct","max"),
    )
)

robustness_summary["transfer_rate_pct"]=100*robustness_summary["transfer_rate"]

display(robustness_summary)
robustness_summary.to_csv(
    OUT/"robustness_transfer_rate_recalibrated.csv",index=False
)

# Parte B — Stress test com \(\theta\) da margem de referência

## 9. Aplicação de \(\widehat\theta_{M=1.00}\) em margens alternativas

Aqui **não há recalibração** da representação quando \(M\) muda.

In [ ]:
reference_theta=(
    freezes[np.isclose(freezes["margin"],1.0)]
    .set_index("replication_id")
)

stress_rows=[]

for rep in replications.itertuples(index=False):
    rid=rep.replication_id
    cache=prediction_cache[rid]

    for margin in CAPACITY_MARGINS:
        scale=(
            TOTAL_CONVENTIONAL_CAPACITY /
            (float(margin)*cache["dev_mean_actual"])
        )

        tst=cache["test"].copy()
        for c in ["actual_peak","ENTSOE","SARIMAX","LSTM"]:
            tst[c]=tst[c]*scale

        for model in ["ENTSOE","SARIMAX","LSTM"]:
            theta_ref=float(reference_theta.loc[rid,f"theta_{model}"])

            for theta,strategy in [(0.0,"baseline"),(theta_ref,"FAME-DOC-refM1")]:
                for row in tst.itertuples(index=False):
                    rr=evaluate_decision(
                        row.date,model,getattr(row,model),row.actual_peak,float(theta)
                    )
                    rr["replication_id"]=rid
                    rr["margin"]=float(margin)
                    rr["strategy"]=strategy
                    rr["theta_reference"]=theta_ref
                    stress_rows.append(rr)

stress=pd.DataFrame(stress_rows)

stress_summary=(
    stress.groupby(
        ["replication_id","margin","model","strategy"],as_index=False
    )
    .agg(mean_loss=("realized_loss","mean"))
)

stress_wide=stress_summary.pivot(
    index=["replication_id","margin","model"],
    columns="strategy",
    values="mean_loss"
).reset_index()

stress_wide["gain"]=(
    stress_wide["baseline"]-stress_wide["FAME-DOC-refM1"]
)
stress_wide["gain_pct"]=100*stress_wide["gain"]/stress_wide["baseline"]
stress_wide["transferred"]=stress_wide["gain"]>0

stress_rate=(
    stress_wide.groupby(["margin","model"],as_index=False)
    .agg(
        transfer_rate=("transferred","mean"),
        mean_gain_pct=("gain_pct","mean"),
        median_gain_pct=("gain_pct","median"),
        min_gain_pct=("gain_pct","min"),
    )
)

stress_rate["transfer_rate_pct"]=100*stress_rate["transfer_rate"]

display(stress_rate)

stress_wide.to_csv(OUT/"stress_test_reference_theta_by_replication.csv",index=False)
stress_rate.to_csv(OUT/"stress_test_reference_theta_summary.csv",index=False)

## 10. Estabilidade de θ em função da margem

In [ ]:
theta_long=[]

for row in freezes.itertuples(index=False):
    for model in ["ENTSOE","SARIMAX","LSTM"]:
        theta_long.append({
            "replication_id":row.replication_id,
            "margin":row.margin,
            "model":model,
            "theta_doc":getattr(row,f"theta_{model}")
        })

theta_long=pd.DataFrame(theta_long)

theta_margin_summary=(
    theta_long.groupby(["margin","model"],as_index=False)
    .agg(
        mean_theta=("theta_doc","mean"),
        median_theta=("theta_doc","median"),
        sd_theta=("theta_doc","std"),
        min_theta=("theta_doc","min"),
        max_theta=("theta_doc","max"),
    )
)

display(theta_margin_summary)
theta_margin_summary.to_csv(OUT/"theta_stability_by_margin.csv",index=False)

## 11. Gráficos de robustez

In [ ]:
for model in ["ENTSOE","SARIMAX","LSTM"]:
    sub=robustness_summary[robustness_summary.model==model].sort_values("margin")

    plt.figure(figsize=(7.5,4.5))
    plt.plot(sub["margin"],sub["transfer_rate_pct"],marker="o")
    plt.ylim(-2,102)
    plt.xlabel("Margem média de capacidade")
    plt.ylabel("Transfer rate (%)")
    plt.title(f"DOC recalibrado — {model}")
    plt.grid(alpha=0.2)
    plt.show()

for model in ["ENTSOE","SARIMAX","LSTM"]:
    sub=stress_rate[stress_rate.model==model].sort_values("margin")

    plt.figure(figsize=(7.5,4.5))
    plt.plot(sub["margin"],sub["transfer_rate_pct"],marker="o")
    plt.ylim(-2,102)
    plt.xlabel("Margem média de capacidade")
    plt.ylabel("Transfer rate (%)")
    plt.title(rf"$\theta$ congelado em M=1.00 — {model}")
    plt.grid(alpha=0.2)
    plt.show()

In [ ]:
for model in ["ENTSOE","SARIMAX","LSTM"]:
    sub=robustness_summary[robustness_summary.model==model].sort_values("margin")

    plt.figure(figsize=(7.5,4.5))
    plt.axhline(0,linestyle="--",linewidth=1)
    plt.plot(sub["margin"],sub["mean_gain_pct"],marker="o")
    plt.xlabel("Margem média de capacidade")
    plt.ylabel("Ganho FAME médio no Test (%)")
    plt.title(f"Robustez do ganho operacional — {model}")
    plt.grid(alpha=0.2)
    plt.show()

# 12. Critério para interpretação científica

A conclusão de robustez deve distinguir dois níveis.

### Robustez do mecanismo DOC

Há evidência forte se SARIMAX/LSTM mantiverem alta taxa de transferência quando
\(\theta\) é recalibrado corretamente em cada cenário de margem.

### Robustez da representação aprendida

A evidência é ainda mais forte se os \(\theta\) aprendidos em \(M=1.00\) continuarem
gerando ganho fora dessa condição quando aplicados sem recalibração.

Resultados como

\[
9/9
\]

em uma única margem são interessantes, mas resultados semelhantes em múltiplos
regimes de capacidade tornam muito menos plausível que o efeito decorra apenas de uma
especificação operacional excessivamente apertada.

Esta análise não altera a camada preditiva nem escolhe cenários com base nos resultados.
As margens são fixadas previamente em

\[
1.00,\;1.10,\;1.20,\;1.30,\;1.40.
\]